# 04 — Baseline forecasts (14-day holdout)

**Series:** hourly `load_mw` (`data/raw/de_lu_load_hourly.parquet`).

**Test set:** the final **14 × 24 = 336** hours. All forecasts use only history **strictly before** each test timestamp (extended history from the train period for seasonal lags).

**Baselines**
- **Naïve:** last observed value $\hat{y}_t = y_{t-1}$
- **Seasonal naïve (lag 24):** same hour yesterday $\hat{y}_t = y_{t-24}$
- **Seasonal naïve (lag 168):** same hour last week $\hat{y}_t = y_{t-168}$
- **Drift:** $\hat{y}_{T+h} = y_T + h\,(y_T - y_1)/(T-1)$ with train length $T$ and horizon $h=1,\ldots$ into the holdout (`y_1`,`y_T` are first and last train values).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = Path(r"C:\Users\mhmta\Cursor_Study\energy-ts-fundamentals\data\raw\de_lu_load_hourly.parquet")
if not DATA_PATH.is_file():
    DATA_PATH = Path("data/raw/de_lu_load_hourly.parquet")

HOLDOUT_DAYS = 14 #  the number of days to hold out for testing

load_hourly = pd.read_parquet(DATA_PATH)
idx = load_hourly.index
if not isinstance(idx, pd.DatetimeIndex): # function in Python is used to check if an object is an instance of a specified class or a subclass thereof.
    load_hourly.index = pd.to_datetime(idx, utc=True)
if load_hourly.index.tz is None:
    load_hourly.index = load_hourly.index.tz_localize(
        "Europe/Berlin", ambiguous="infer", nonexistent="shift_forward"
    )
else:
    load_hourly.index = load_hourly.index.tz_convert("Europe/Berlin")

y = (
    load_hourly["load_mw"]
    .sort_index()
    .asfreq("1h")
    .interpolate(limit_direction="both")
)

n_test = HOLDOUT_DAYS * 24
if len(y) <= n_test + 168:
    raise ValueError(f"Need more than {n_test + 168} hours for train + test + lag-168.")

train = y.iloc[:-n_test]
test = y.iloc[-n_test:]
y_all = y.values.astype(float) #  first extracts the numerical data from the pandas Series y as a NumPy array using .values,

print("Rows:", len(y), "| Train:", len(train), "| Test:", len(test))
print("Test window:", test.index.min(), "->", test.index.max())

In [ ]:
n = len(y_all)
base = n - n_test  # index of first test hour in full series

# Naive: lag-1
pred_naive = y_all[base - 1 : n - 1]

# Seasonal naive: lag-24 and lag-168
pred_sn24 = y_all[base - 24 : n - 24]
pred_sn168 = y_all[base - 168 : n - 168]

# Drift: Hyndman & Athanasopoulos — linear trend from first to last train observation
T_train = len(train)
y_first = float(train.iloc[0])
y_last = float(train.iloc[-1])
if T_train < 2:
    raise ValueError("Drift needs at least two train points.")
slope = (y_last - y_first) / (T_train - 1)
horizons = np.arange(1, n_test + 1, dtype=float)
pred_drift = y_last + horizons * slope

y_test = test.values.astype(float)
assert pred_naive.shape == y_test.shape == pred_sn24.shape == pred_sn168.shape == pred_drift.shape

In [ ]:
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float: #  the Mean Absolute Percentage Error (MAPE) 
    # a lower percentage indicates higher accuracy—but it can be misleading with low-demand data
    eps = np.finfo(float).eps # np.finfo(float).eps is the smallest representable positive number for a float.
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100.0)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float: # Symmetric Mean Absolute Percentage Error (sMAPE) function
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > np.finfo(float).eps # The mask identifies where the denominator is greater than this small number, which helps to avoid division by zero errors.
    return float(np.mean(200.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask])) #  Multiplies the result by 200.0, as the sMAPE formula typically involves multiplying by 100% and then by 2 to account for the sum of absolute values in the denominator.
    # MAPE becomes undefined or extremely large when the actual value (y_true) is zero or very close to zero, leading to division by zero or a very large error. sMAPE mitigates this by using the average of the absolute true and predicted values in the denominator

def score_block(name: str, y_pred: np.ndarray) -> pd.Series:
    return pd.Series(
        {
            "method": name,
            "MAE": mae(y_test, y_pred),
            "RMSE": rmse(y_test, y_pred),
            "MAPE_%": mape(y_test, y_pred),
            "sMAPE_%": smape(y_test, y_pred),
        }
    )


results = pd.DataFrame(
    [
        score_block("Naive (lag-1)", pred_naive),
        score_block("Seasonal naive lag-24", pred_sn24),
        score_block("Seasonal naive lag-168", pred_sn168),
        score_block("Drift", pred_drift),
    ]
).set_index("method")

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
results